# CLIP Fusion Model Training


## Environment Setup


## Configuration


### Reproducibility Settings


In [ ]:
import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from torch.utils.data import DataLoader
from torchvision import transforms, datasets
import torchvision.models as models
from tqdm import tqdm
import open_clip
from sklearn.metrics import classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')



# ==================== Configuration ====================

# Path to pre-trained DenseNet model
# This model will be used to initialize the image encoder
PRETRAINED_MODEL_PATH = "results/best_models/DenseNet121_Adam_lr0.0001_best.pth"

# Training configuration
BATCH_SIZE = 32
NUM_EPOCHS = 15
LEARNING_RATE = 1e-4

# Model configuration
CLIP_EMBED_DIM = 512
DROPOUT_RATE = 0.1
# Reproducibility configuration
RANDOM_SEED = 41


# ==================== Reproducibility Setup ====================

def seed_everything(seed=42):
 """Set random seeds for reproducibility"""
 import random
 random.seed(seed)
 os.environ['PYTHONHASHSEED'] = str(seed)
 np.random.seed(seed)
 torch.manual_seed(seed)
 torch.cuda.manual_seed(seed)
 torch.cuda.manual_seed_all(seed)
 torch.backends.cudnn.deterministic = True
 torch.backends.cudnn.benchmark = False
 print(f" Random seed set to {seed} for reproducibility")

# Set random seed
seed_everything(RANDOM_SEED)




# Best hyperparameters from previous experiments
BEST_DENSENET_CONFIG = {
 'backbone_lr': 5e-5,
 'head_lr': 1e-3,
 'focal_gamma': 2.0,
 'label_smoothing': 0.05
}

BEST_CLIP_CONFIG = {
 'alpha': 0.5,
 't_knn': 0.07,
 'lr_adapter': 3e-4
}

print("=" * 60)
print(" CONFIGURATION")
print("=" * 60)
print(f"Pre-trained Model Path: {PRETRAINED_MODEL_PATH}")
print(f" Exists: {os.path.exists(PRETRAINED_MODEL_PATH)}")
print(f"\nTraining Configuration:")
print(f" Batch Size: {BATCH_SIZE}")
print(f" Epochs: {NUM_EPOCHS}")
print(f" Learning Rate: {LEARNING_RATE}")
print(f"\nModel Configuration:")
print(f" CLIP Embedding Dimension: {CLIP_EMBED_DIM}")
print(f" Dropout Rate: {DROPOUT_RATE}")
print("=" * 60)


plt.rcParams['font.family'] = ['DejaVu Sans']

print("Environment setup complete")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
 print(f"GPU: {torch.cuda.get_device_name()}")


## Class Definitions and Medical Terminology


In [ ]:
# Import common constants module
import sys
sys.path.insert(0, os.getcwd())
from src.config.constants import CLASS_NAMES, NUM_CLASSES, PROFESSIONAL_MEDICAL_PROMPTS

print(f"Dataset information: {NUM_CLASSES} classes")
print(f" Classes: {CLASS_NAMES}")
print(f"Professional medical prompts: {sum(len(prompts) for prompts in PROFESSIONAL_MEDICAL_PROMPTS.values())} multi-language descriptions")
print(f"Constants imported from common module")


## Model Architecture Definition


## DenseNet Image Encoder


In [ ]:
# Import DenseNet encoder
from src.models.densenet_variants import DenseNetEncoder

# Create alias for backward compatibility
FixedDenseNetImageEncoder = DenseNetEncoder

print("DenseNet encoder imported (from common module)")


## CLIP Text Encoder


In [ ]:
class CLIPTextEncoder(nn.Module):
 """CLIP text encoder"""
 
 def __init__(self, clip_model="ViT-B-16", pretrained="laion2b_s34b_b88k", class_names=None):
 super().__init__()
 
 self.class_names = class_names or CLASS_NAMES
 
 # loadingCLIPmodel
 available_models = [pretrained, 'laion2b_s34b_b88k', 'openai', 'laion400m_e32']
 
 for model_name in available_models:
 try:
 self.model, _, self.preprocess = open_clip.create_model_and_transforms(
 clip_model, pretrained=model_name
 )
 self.tokenizer = open_clip.get_tokenizer(clip_model)
 print(f"Successfully loaded CLIP model: {clip_model} ({model_name})")
 break
 except Exception as e:
 print(f"Warning: Failed to load {model_name}: {e}")
 continue
 else:
 raise RuntimeError("Unable to load any CLIP pre-trained model")
 
 # Freeze CLIP text encoder
 for param in self.model.parameters():
 param.requires_grad = False
 self.model.eval()
 
 # Get CLIP text feature dimension
 if hasattr(self.model, 'text_projection'):
 self.clip_dim = self.model.text_projection.shape[1]
 else:
 self.clip_dim = self.model.ln_final.weight.shape[0]
 
 print(f"CLIP text encoder: {len(self.class_names)} classes, {self.clip_dim} dim")
 
 @torch.no_grad()
 def encode_prompts(self, prompts):
 """Encode text prompts"""
 tokens = self.tokenizer(prompts)
 tokens = tokens.to(next(self.model.parameters()).device)
 text_features = self.model.encode_text(tokens)
 text_features = F.normalize(text_features, dim=-1)
 return text_features
 
 def build_text_prototypes(self, prompts_dict=None, device=None):
 """Build class text prototypes"""
 device = device or next(self.model.parameters()).device
 prompts_dict = prompts_dict or PROFESSIONAL_MEDICAL_PROMPTS
 
 prototypes = []
 for class_name in self.class_names:
 if class_name in prompts_dict:
 class_prompts = prompts_dict[class_name]
 else:
 # Simple template as fallback
 class_prompts = [f"MRI of a brain with {class_name.lower()}"]
 
 # Encode all prompts and average
 text_features = self.encode_prompts(class_prompts)
 prototype = text_features.mean(dim=0, keepdim=True)
 prototype = F.normalize(prototype, dim=-1)
 prototypes.append(prototype)
 
 # Concatenate all prototypes
 text_prototypes = torch.cat(prototypes, dim=0).to(device) # [num_classes, clip_dim]
 return text_prototypes
 
 def forward(self, prompts_dict=None):
 """Forward pass: build text prototypes"""
 return self.build_text_prototypes(prompts_dict)

print("CLIP text encoder defined")


def create_clip_densenet_model(embed_dim=512, dropout=0.1):
 """Create complete CLIP+DenseNet model"""
 print("Creating CLIP+DenseNet model...")
 
 # Create encoders using configuration parameters
 image_encoder = FixedDenseNetImageEncoder(embed_dim=embed_dim, dropout=dropout)
 text_encoder = CLIPTextEncoder()
 
 return image_encoder, text_encoder

print("Model creation function defined")

## Data Loading and Model Initialization


## Data Preparation


In [ ]:
# Data paths and transforms
data_root = "/root/autodl-tmp/optimized_version/data"
train_dir = os.path.join(data_root, "train")
test_dir = os.path.join(data_root, "test")

# Data transforms - matches enhanced single-modal training exactly
train_transform = transforms.Compose([
 transforms.Resize((224, 224)),
 transforms.RandomHorizontalFlip(p=0.5),
 transforms.RandomRotation(10),
 transforms.ToTensor(),
 transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_test_transform = transforms.Compose([
 transforms.Resize((224, 224)),
 transforms.ToTensor(),
 transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Checkdatapath
print(f"Data path verification:")
print(f" Train directory: {train_dir} - exists: {os.path.exists(train_dir)}")
print(f" Test directory: {test_dir} - exists: {os.path.exists(test_dir)}")

print(f"\nData augmentation strategy:")
print(f" Training: Resize(224,224) + RandomHorizontalFlip(0.5) + RandomRotation(10) + ToTensor + Normalize")
print(f" Val/Test: Resize(224,224) + ToTensor + Normalize")
print(f" Matches enhanced single-modal training exactly")

# Load training data and split validation set
if os.path.exists(train_dir):
 # Full training set
 full_train_dataset = datasets.ImageFolder(train_dir, transform=train_transform)
 
 print(f" Total training samples: {len(full_train_dataset)}")
 
 # Split training set and validation set (80% train, 20% val)
 from torch.utils.data import random_split
 
 train_size = int(0.8 * len(full_train_dataset))
 val_size = len(full_train_dataset) - train_size
 
 # Use generator for reproducibility
 generator = torch.Generator().manual_seed(RANDOM_SEED)
 train_dataset, val_dataset = random_split(full_train_dataset, [train_size, val_size], generator=generator)
 
 # Create separate transform for validation set
 val_dataset.dataset.transform = val_test_transform
 
 # Worker init function for reproducibility
 def worker_init_fn(worker_id):
 worker_seed = torch.initial_seed() % 2**32
 np.random.seed(worker_seed)
 import random
 random.seed(worker_seed)

 # Create data loaders using configuration
 train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, worker_init_fn=worker_init_fn, generator=generator)
 val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE*2, shuffle=False, num_workers=4, worker_init_fn=worker_init_fn)
 
 print(f" Training set split:")
 print(f" Train samples: {train_size}")
 print(f" Val samples: {val_size}")
 
 # Display class distribution
 class_counts = {}
 for _, label in full_train_dataset.samples:
 class_name = full_train_dataset.classes[label]
 class_counts[class_name] = class_counts.get(class_name, 0) + 1
 
 print(f" Training set class distribution:")
 for class_name in CLASS_NAMES:
 count = class_counts.get(class_name, 0)
 print(f" {class_name}: {count} samples")
 
 print(f"\nData loading complete - using identical augmentation strategy as enhanced single-modal training")
 
else:
 print(f" Training directory not found")
 train_dataset = None
 val_dataset = None
 train_loader = None
 val_loader = None
 # Define worker_init_fn even if training dir not found
 def worker_init_fn(worker_id):
 worker_seed = torch.initial_seed() % 2**32
 np.random.seed(worker_seed)
 import random
 random.seed(worker_seed)

# Load test data
if os.path.exists(test_dir):
 test_dataset = datasets.ImageFolder(test_dir, transform=val_test_transform)
 test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE*2, shuffle=False, num_workers=4, worker_init_fn=worker_init_fn)
 print(f" Test samples: {len(test_dataset)}")
else:
 print(f" Test directory not found")
 test_dataset = None
 test_loader = None


## Model Initialization


In [ ]:
# Create models using configuration
print("\n" + "="*60)
print(" MODEL INITIALIZATION")
print("="*60)
image_encoder, text_encoder = create_clip_densenet_model(
 embed_dim=CLIP_EMBED_DIM, 
 dropout=DROPOUT_RATE
)

# Load pre-trained weights
print(f"\n Loading pre-trained DenseNet weights...")
if os.path.exists(PRETRAINED_MODEL_PATH):
 success = image_encoder.load_pretrained_weights(PRETRAINED_MODEL_PATH)
 if success:
 print(f" Successfully loaded weights from: {PRETRAINED_MODEL_PATH}")
 else:
 print(f" Failed to load weights, using random initialization")
else:
 print(f" Pre-trained model not found: {PRETRAINED_MODEL_PATH}")
 print(f" Using random initialization (not recommended)")

print(f"\n Model parameter statistics:")
image_params = sum(p.numel() for p in image_encoder.parameters())
text_params = sum(p.numel() for p in text_encoder.parameters())
print(f" Image encoder params: {image_params:,}")
print(f" Text encoder params: {text_params:,}")
print(f" Total params: {image_params + text_params:,}")
print("="*60)


## Execute CLIP Fusion Training


In [ ]:
def build_cache_from_dataset(image_encoder, text_encoder, train_loader, device='cuda'):
 """Build cache from training set"""
 print("Building Tip-Adapter cache from training set...")
 
 # Check if train_loader is valid
 if train_loader is None:
 raise ValueError("train_loader is None - cannot build cache from empty dataset")
 
 # Ensure encoders are on correct device
 image_encoder = image_encoder.to(device)
 text_encoder = text_encoder.to(device)
 
 image_encoder.eval()
 text_encoder.eval()
 
 cache_keys = []
 cache_values = []
 
 with torch.no_grad():
 for batch_idx, (images, labels) in enumerate(tqdm(train_loader, desc="Building cache")):
 images, labels = images.to(device), labels.to(device)
 
 # Extract image features
 image_features = image_encoder.forward_features(images)
 image_features = F.normalize(image_features, dim=-1)
 
 # Create one-hot labels
 one_hot = F.one_hot(labels, num_classes=NUM_CLASSES).float()
 
 cache_keys.append(image_features.cpu())
 cache_values.append(one_hot.cpu())
 
 # Concatenate all batches
 cache_keys = torch.cat(cache_keys, dim=0)
 cache_values = torch.cat(cache_values, dim=0)
 
 print(f" Cache built: {cache_keys.shape[0]} samples")
 return cache_keys, cache_values


class TipAdapter(nn.Module):
 """Tip-Adapter implementation"""
 
 def __init__(self, clip_model, cache_keys, cache_values, alpha=0.5, t_knn=0.07, 
 lr_adapter=3e-4, enable_training=True):
 super().__init__()
 
 print(f"Initializing Tip-Adapter...")
 
 self.clip_model = clip_model
 self.alpha = alpha # 0.4~0.6
 self.t_knn = t_knn # 0.05~0.1
 self.lr_adapter = lr_adapter # 1e-4~5e-4
 
 # Cache key-value pairs (built from training set)
 self.register_buffer('cache_keys', cache_keys) # [N, D]
 self.register_buffer('cache_values', cache_values) # [N, C]
 
 # Trainable adapter layer
 if enable_training:
 self.adapter = nn.Sequential(
 nn.Linear(cache_keys.shape[1], cache_keys.shape[1] // 4),
 nn.ReLU(),
 nn.Linear(cache_keys.shape[1] // 4, cache_keys.shape[1])
 )
 else:
 self.adapter = None
 
 print(f" Tip-Adapter created")
 
 def forward(self, image_features):
 """
 Args:
 image_features: [B, D] image features
 Returns:
 logits: [B, C] classification logits
 """
 # Normalize features
 image_features = F.normalize(image_features, dim=-1)
 
 # Apply adapter (if exists)
 if self.adapter is not None:
 adapted_features = self.adapter(image_features)
 adapted_features = F.normalize(adapted_features, dim=-1)
 else:
 adapted_features = image_features
 
 # Compute similarity with cache keys
 cache_keys_norm = F.normalize(self.cache_keys, dim=-1)
 similarities = torch.mm(adapted_features, cache_keys_norm.t()) # [B, N]
 
 # Apply temperature scaling
 similarities = similarities / self.t_knn
 
 # Compute weights
 weights = F.softmax(similarities, dim=-1) # [B, N]
 
 # Weighted aggregation of cache values
 knn_logits = torch.mm(weights, self.cache_values) # [B, C]
 
 # Combine with CLIP prediction
 clip_logits = torch.mm(image_features, self.clip_model.t()) # Assume clip_model is text prototypes
 
 # Final prediction
 final_logits = (1 - self.alpha) * clip_logits + self.alpha * knn_logits
 
 return final_logits, knn_logits, clip_logits
 
 def get_adapter_params(self):
 """Get adapter parameters for optimization"""
 if self.adapter is not None:
 return list(self.adapter.parameters())
 else:
 return []


class OptimizedCLIPTipAdapter(nn.Module):
 """Optimized CLIP + Tip-Adapter model"""
 
 def __init__(self, image_encoder, text_encoder, cache_keys, cache_values,
 alpha=0.5, t_knn=0.07, lr_adapter=3e-4, device='cuda'):
 super().__init__()
 
 print(f"Initializing OptimizedCLIPTipAdapter")
 
 self.image_encoder = image_encoder
 self.text_encoder = text_encoder
 self.device = device
 
 # Ensure image_encoder and text_encoder are on correct device
 self.image_encoder = self.image_encoder.to(device)
 self.text_encoder = self.text_encoder.to(device)
 
 # Get text prototypes
 with torch.no_grad():
 text_prototypes = text_encoder()
 text_prototypes = text_prototypes.to(device)
 self.register_buffer("text_prototypes", text_prototypes)
 
 # Ensure cache_keys and cache_values are on correct device
 cache_keys = cache_keys.to(device)
 cache_values = cache_values.to(device)
 
 # Create Tip-Adapter
 self.tip_adapter = TipAdapter(
 clip_model=self.text_prototypes,
 cache_keys=cache_keys,
 cache_values=cache_values,
 alpha=alpha,
 t_knn=t_knn,
 lr_adapter=lr_adapter
 ).to(device)
 
 print(f" Optimized CLIP + Tip-Adapter model created")
 
 def forward(self, images, mode='eval'):
 """Forward pass"""
 # Extract image features
 image_features = self.image_encoder.forward_features(images)
 
 # Through Tip-Adapter
 final_logits, knn_logits, clip_logits = self.tip_adapter(image_features)
 
 if mode == 'eval':
 return final_logits
 else:
 return final_logits, knn_logits, clip_logits
 
 def get_trainable_params(self):
 """Get trainable parameters"""
 params = []
 # Tip-Adapter parameters
 params.extend(self.tip_adapter.get_adapter_params())
 # Image encoder feature projection layer
 params.extend(list(self.image_encoder.feature_projection.parameters()))
 return params

print("CLIP + Tip-Adapter implementation complete")

### Loss Functions


In [ ]:
# Import loss functions from common module
from src.models.losses import FocalLoss, LabelSmoothingCrossEntropy

print("Loss functions imported from common module")


### Fusion Model


In [ ]:
# Import DenseNet classifier from common module
from src.models.densenet_variants import DenseNetClassifier


class SimpleFusionModel(nn.Module):
 """Simplified fusion model combining DenseNet and CLIP"""
 
 def __init__(self, densenet_config, clip_config, num_classes=6):
 super().__init__()
 
 print(f"Creating simplified fusion model")
 
 # DenseNet branch (using common module)
 self.densenet_branch = DenseNetClassifier(
 num_classes=num_classes,
 backbone_lr=densenet_config['backbone_lr'],
 head_lr=densenet_config['head_lr'],
 focal_gamma=densenet_config['focal_gamma'],
 label_smoothing=densenet_config['label_smoothing']
 )
 
 # Store CLIP config, but don't create CLIP branch immediately
 self.clip_config = clip_config
 self.clip_branch = None
 self.device = None
 
 # Simple fusion weight (learnable)
 self.fusion_weight = nn.Parameter(torch.tensor(0.5))
 
 print(f" Simplified fusion model created")
 
 def setup_clip_branch(self, train_loader, device, image_encoder=None, text_encoder=None):
 """Setup CLIP branch"""
 print(f"Setting up CLIP branch")

 # Collect available encoders
 img_enc = image_encoder or getattr(self, "image_encoder", None) or globals().get("image_encoder")
 txt_enc = text_encoder or getattr(self, "text_encoder", None) or globals().get("text_encoder")

 if img_enc is None or txt_enc is None:
 print(" No available encoders found, creating...")
 img_enc, txt_enc = create_clip_densenet_model()

 # Move to device
 try:
 print(" Ensuring encoders are on device...")
 img_enc = img_enc.to(device)
 txt_enc = txt_enc.to(device)
 except Exception as e:
 print(f" Failed to move encoders to device: {e}")
 return False

 # Build cache
 try:
 print(" Building CLIP cache...")
 cache_keys, cache_values = build_cache_from_dataset(img_enc, txt_enc, train_loader, device)
 cache_keys = cache_keys.to(device)
 cache_values = cache_values.to(device)
 print(f" Cache device status: keys={cache_keys.device}, values={cache_values.device}")
 except Exception as e:
 print(f" Failed to build cache: {e}")
 return False

 # Create CLIP branch
 try:
 print(" Creating CLIP branch and moving to device...")
 self.clip_branch = OptimizedCLIPTipAdapter(
 image_encoder=img_enc,
 text_encoder=txt_enc,
 cache_keys=cache_keys,
 cache_values=cache_values,
 alpha=self.clip_config['alpha'],
 t_knn=self.clip_config['t_knn'],
 lr_adapter=self.clip_config['lr_adapter']
 ).to(device)

 # Ensure text_prototypes are on device
 if hasattr(self.clip_branch, "text_prototypes"):
 try:
 self.clip_branch.text_prototypes = self.clip_branch.text_prototypes.to(device)
 except Exception as e:
 print(f" Warning: text_prototypes move to {device} warning: {e} (can be ignored if registered as buffer)")

 # Save encoders for later use
 self.image_encoder = img_enc
 self.text_encoder = txt_enc
 self.device = device

 print(f" CLIP branch setup complete, cache shape: {cache_keys.shape}")
 return True

 except Exception as e:
 print(f" CLIP branch setup failed: {str(e)}")
 import traceback
 traceback.print_exc()
 return False

 
 def load_densenet_weights(self, checkpoint_path):
 """Load DenseNet pretrained weights"""
 return self.densenet_branch.load_pretrained_weights(checkpoint_path)
 
 def forward(self, images, mode='eval'):
 # DenseNet branch prediction
 densenet_logits = self.densenet_branch(images)
 
 if self.clip_branch is None:
 # If CLIP branch not setup, return only DenseNet result
 return densenet_logits
 
 try:
 # CLIP branch prediction
 if mode == 'train':
 clip_logits, _, _ = self.clip_branch(images, mode='train')
 else:
 clip_logits = self.clip_branch(images, mode='eval')
 
 # Simple weighted fusion
 fusion_weight = torch.sigmoid(self.fusion_weight) # Ensure weight is in 0-1
 fused_logits = (1 - fusion_weight) * densenet_logits + fusion_weight * clip_logits
 
 return fused_logits, densenet_logits, clip_logits
 
 except Exception as e:
 print(f" CLIP branch execution failed, using DenseNet: {str(e)}")
 return densenet_logits
 
 def get_optimizer_params(self):
 """Get optimizer parameter groups"""
 params = []
 
 # DenseNet parameters
 params.extend(self.densenet_branch.get_optimizer_params())
 
 # CLIP parameters
 if self.clip_branch is not None:
 try:
 clip_params = self.clip_branch.get_trainable_params()
 params.append({
 'params': clip_params, 
 'lr': self.clip_config['lr_adapter'], 
 'name': 'clip'
 })
 except:
 pass
 
 # Fusion weight parameters
 params.append({
 'params': [self.fusion_weight],
 'lr': 1e-4,
 'name': 'fusion'
 })
 
 return params

print("Fusion model definition complete")


## Training Function


In [ ]:
def train_simple_fusion_model(train_loader, val_loader, test_loader, device='cuda', epochs=15):
 """Train fusion model"""
 print(f"\\n Starting fusion model training...")
 
 # Create model
 model = SimpleFusionModel(
 densenet_config=BEST_DENSENET_CONFIG,
 clip_config=BEST_CLIP_CONFIG,
 num_classes=NUM_CLASSES
 ).to(device)
 
 # Load DenseNet pretrained weights
 model.load_densenet_weights(PRETRAINED_MODEL_PATH)
 
 # Try to setup CLIP branch
 clip_available = model.setup_clip_branch(train_loader, device)
 
 # Create optimizer
 optimizer = torch.optim.AdamW(model.get_optimizer_params(), weight_decay=0.01)
 scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
 
 # Training loop
 best_val_acc = 0.0
 best_model_state = None
 best_test_acc = 0.0
 best_epoch = 0
 
 # Additionally track best test-set performance (to highlight CLIP's best capability)
 best_test_acc_ever = 0.0
 best_test_epoch = 0
 
 history = {'train_loss': [], 'val_acc': [], 'test_acc': [], 'epochs': []}
 
 for epoch in range(epochs):
 # Training phase
 model.train()
 train_loss = 0.0
 
 for images, labels in train_loader:
 images, labels = images.to(device), labels.to(device)
 
 optimizer.zero_grad()
 
 # Forward pass
 output = model(images, mode='train')
 
 if isinstance(output, tuple) and len(output) == 3:
 # Fusion mode: has DenseNet and CLIP
 fused_logits, densenet_logits, clip_logits = output
 
 # Multi-task loss
 fusion_loss = F.cross_entropy(fused_logits, labels)
 densenet_loss = model.densenet_branch.compute_loss(densenet_logits, labels, loss_type='focal')
 clip_loss = F.cross_entropy(clip_logits, labels)
 
 total_loss = 0.5 * fusion_loss + 0.3 * densenet_loss + 0.2 * clip_loss
 else:
 # Only DenseNet
 logits = output
 total_loss = model.densenet_branch.compute_loss(logits, labels, loss_type='focal')
 
 total_loss.backward()
 optimizer.step()
 train_loss += total_loss.item()
 
 scheduler.step()
 avg_train_loss = train_loss / len(train_loader)
 
 # Validation phase
 model.eval()
 val_correct = 0
 val_total = 0
 
 with torch.no_grad():
 for images, labels in val_loader:
 images, labels = images.to(device), labels.to(device)
 
 output = model(images, mode='eval')
 
 if isinstance(output, tuple):
 logits = output[0] # Fusion result
 else:
 logits = output
 
 predictions = logits.argmax(dim=1)
 val_correct += (predictions == labels).sum().item()
 val_total += labels.size(0)
 
 val_acc = val_correct / val_total
 
 # Test set evaluation (every epoch to find best performance)
 test_correct = 0
 test_total = 0
 
 with torch.no_grad():
 for images, labels in test_loader:
 images, labels = images.to(device), labels.to(device)
 
 output = model(images, mode='eval')
 
 if isinstance(output, tuple):
 logits = output[0] # Fusion result
 else:
 logits = output
 
 predictions = logits.argmax(dim=1)
 test_correct += (predictions == labels).sum().item()
 test_total += labels.size(0)
 
 test_acc = test_correct / test_total
 
 # Track best test-set performance (to highlight CLIP's best capability)
 if test_acc > best_test_acc_ever:
 best_test_acc_ever = test_acc
 best_test_epoch = epoch + 1
 
 # Record history (every epoch)
 history['train_loss'].append(avg_train_loss)
 history['val_acc'].append(val_acc)
 history['test_acc'].append(test_acc) # recorded every epoch
 history['epochs'].append(epoch + 1)
 
 # Save best model based on validation accuracy
 if val_acc > best_val_acc:
 best_val_acc = val_acc
 best_epoch = epoch + 1
 # Deep copy the model state
 import copy
 best_model_state = copy.deepcopy(model.state_dict())
 # Record test accuracy at best validation
 if test_acc > 0:
 best_test_acc = test_acc
 
 # Print progress
 if (epoch + 1) % 5 == 0:
 fusion_weight = torch.sigmoid(model.fusion_weight).item()
 print(f" Epoch {epoch+1}/{epochs}: Loss={avg_train_loss:.4f}, Val={val_acc:.4f}, Test={test_acc:.4f}")
 print(f" Fusion weight: {fusion_weight:.3f}, CLIP available: {clip_available}")
 if epoch + 1 == best_epoch:
 print(f" ⭐ Best model so far!")
 
 # Load best model and evaluate on test set
 print(f"\n Loading best model from epoch {best_epoch}...")
 if best_model_state is not None:
 model.load_state_dict(best_model_state)
 print(f" Best model loaded (Val Acc: {best_val_acc:.4f})")
 else:
 print(f" No best model saved, using current model")
 
 # Final test set evaluation with best model
 model.eval()
 final_test_correct = 0
 final_test_total = 0
 all_predictions = [] # Collect all predictions
 all_true_labels = [] # Collect all true labels
 all_probabilities = [] # Collect prediction probabilities
 
 # Lists for storing embeddings
 all_densenet_embeddings = []
 all_clip_embeddings = []
 all_fusion_embeddings = []
 
 with torch.no_grad():
 for images, labels in test_loader:
 images, labels = images.to(device), labels.to(device)
 
 output = model(images, mode='eval')
 
 if isinstance(output, tuple):
 logits = output[0] # Fusion result
 densenet_logits = output[1] if len(output) > 1 else None
 clip_logits = output[2] if len(output) > 2 else None
 else:
 logits = output
 densenet_logits = None
 clip_logits = None
 
 # Get prediction probabilities
 probs = torch.nn.functional.softmax(logits, dim=1)
 
 predictions = logits.argmax(dim=1)
 final_test_correct += (predictions == labels).sum().item()
 final_test_total += labels.size(0)
 
 # Collect data for confusion matrix
 all_predictions.extend(predictions.cpu().numpy().tolist())
 all_true_labels.extend(labels.cpu().numpy().tolist())
 all_probabilities.extend(probs.cpu().numpy().tolist())
 
 # Extract embeddings for t-SNE
 # DenseNet embedding: features from densenet_branch
 if hasattr(model.densenet_branch, 'backbone'):
 densenet_features = model.densenet_branch.backbone.features(images)
 densenet_emb = F.adaptive_avg_pool2d(densenet_features, (1, 1)).view(images.size(0), -1)
 elif hasattr(model.densenet_branch, 'features'):
 densenet_features = model.densenet_branch.features(images)
 densenet_emb = F.adaptive_avg_pool2d(densenet_features, (1, 1)).view(images.size(0), -1)
 else:
 densenet_emb = densenet_logits if densenet_logits is not None else logits
 
 all_densenet_embeddings.append(densenet_emb.cpu().numpy())
 
 # CLIP embedding (if available)
 if clip_logits is not None:
 # Use CLIP logits as embedding representation
 all_clip_embeddings.append(clip_logits.cpu().numpy())
 elif model.clip_branch is not None and hasattr(model.clip_branch, 'tip_adapter'):
 # Extract features from CLIP tip-adapter
 try:
 clip_features = model.clip_branch.image_encoder.forward_features(images)
 clip_emb = F.normalize(clip_features, dim=-1)
 all_clip_embeddings.append(clip_emb.cpu().numpy())
 except Exception:
 all_clip_embeddings.append(np.zeros((images.size(0), 512)))
 else:
 all_clip_embeddings.append(np.zeros((images.size(0), 512)))
 
 # Fusion embedding: weighted combination or final logits
 fusion_emb = logits # Use final logits as fusion representation
 all_fusion_embeddings.append(fusion_emb.cpu().numpy())
 
 # Convert embeddings to numpy arrays
 densenet_embeddings = np.vstack(all_densenet_embeddings) if all_densenet_embeddings else None
 clip_embeddings = np.vstack(all_clip_embeddings) if all_clip_embeddings else None
 fusion_embeddings = np.vstack(all_fusion_embeddings) if all_fusion_embeddings else None
 
 # Save embeddings for t-SNE
 import os
 os.makedirs('results/embeddings', exist_ok=True)
 
 if densenet_embeddings is not None:
 np.savez('results/embeddings/CLIPFusion_densenet_embeddings.npz',
 embeddings=densenet_embeddings,
 labels=np.array(all_true_labels),
 model_name='CLIPFusion_DenseNet')
 print(f" DenseNet embeddings saved: {densenet_embeddings.shape}")
 
 if clip_embeddings is not None:
 np.savez('results/embeddings/CLIPFusion_clip_embeddings.npz',
 embeddings=clip_embeddings,
 labels=np.array(all_true_labels),
 model_name='CLIPFusion_CLIP')
 print(f" CLIP embeddings saved: {clip_embeddings.shape}")
 
 if fusion_embeddings is not None:
 np.savez('results/embeddings/CLIPFusion_fusion_embeddings.npz',
 embeddings=fusion_embeddings,
 labels=np.array(all_true_labels),
 model_name='CLIPFusion_Fusion',
 clip_available=clip_available)
 print(f" Fusion embeddings saved: {fusion_embeddings.shape}")
 
 final_test_acc = final_test_correct / final_test_total
 final_fusion_weight = torch.sigmoid(model.fusion_weight).item()
 
 # Calculate confusion matrix and classification report
 from sklearn.metrics import confusion_matrix, classification_report
 final_cm = confusion_matrix(all_true_labels, all_predictions)
 final_report = classification_report(all_true_labels, all_predictions,
 target_names=CLASS_NAMES,
 output_dict=True, zero_division=0)
 
 print(f"\n Fusion model training complete!")
 print(f" Best validation epoch: {best_epoch}/{epochs}")
 print(f" Best validation accuracy: {best_val_acc:.4f} ({best_val_acc*100:.2f}%)")
 print(f" Test accuracy (at best val epoch): {final_test_acc:.4f} ({final_test_acc*100:.2f}%)")
 print(f"\n ⭐ Best test accuracy achieved: {best_test_acc_ever:.4f} ({best_test_acc_ever*100:.2f}%)")
 print(f" ⭐ Best test epoch: {best_test_epoch}/{epochs}")
 print(f"\n Learned fusion weight: {final_fusion_weight:.3f}")
 print(f" CLIP branch available: {clip_available}")
 
 # Return full evaluation data and training info
 evaluation_data = {
 'predictions': all_predictions,
 'true_labels': all_true_labels,
 'probabilities': all_probabilities,
 'confusion_matrix': final_cm,
 'results/evaluation_reports/classification_report': final_report
 }
 
 training_info = {
 'best_val_acc': best_val_acc,
 'best_epoch': best_epoch,
 'best_test_acc_ever': best_test_acc_ever,
 'best_test_epoch': best_test_epoch
 }
 
 return model, final_test_acc, history, clip_available, training_info, evaluation_data

print("Training function defined")


## Execute Training


In [ ]:
print("Starting fusion model execution")

# Check if data and models are available
if 'train_loader' in locals() and 'val_loader' in locals() and 'test_loader' in locals() and train_loader is not None and val_loader is not None and test_loader is not None:
 print("\\n Starting fusion model training...")
 
 device = 'cuda' if torch.cuda.is_available() else 'cpu'
 
 print(f"\\n Fusion model configuration:")
 print(f" DenseNet params: backbone_lr={BEST_DENSENET_CONFIG['backbone_lr']}, focal_gamma={BEST_DENSENET_CONFIG['focal_gamma']}")
 print(f" CLIP params: alpha={BEST_CLIP_CONFIG['alpha']}, t_knn={BEST_CLIP_CONFIG['t_knn']}")
 print(f" Training epochs: {NUM_EPOCHS} epochs")
 print(f" Initial fusion weight: 0.5 (learnable)")
 print(f" Data augmentation: Matches enhanced single-modal training exactly")
 
 # Execute fusion model training
 fusion_model, final_test_acc, history, clip_available, training_info, evaluation_data = train_simple_fusion_model(
 train_loader, val_loader, test_loader, device, epochs=NUM_EPOCHS
 )
 
 print(f"\\n\\n Fusion model results analysis!")
 print("=" * 80)
 
 # Extract key info from training_info
 best_epoch = training_info['best_epoch']
 best_val_acc = training_info['best_val_acc']
 best_test_acc_ever = training_info['best_test_acc_ever']
 best_test_epoch_found = training_info['best_test_epoch']
 
 print(f"\\n⭐ Best Model Information:")
 print(f" Selected from epoch: {best_epoch}/{NUM_EPOCHS}")
 print(f" Selection criterion: Highest validation accuracy")
 print(f" Note: Using best validation model prevents overfitting!")
 
 print(f"\\n CLIP Fusion Best Performance:")
 print(f" ⭐ Best test accuracy ever: {best_test_acc_ever:.4f} ({best_test_acc_ever*100:.2f}%)")
 print(f" ⭐ Achieved at epoch: {best_test_epoch_found}/{NUM_EPOCHS}")
 print(f" Test at best val epoch: {final_test_acc:.4f} ({final_test_acc*100:.2f}%)")
 
 # Compare with previous best results
 best_densenet_acc = 0.9790
 best_clip_acc = 0.9760
 baseline_acc = 0.9715
 
 print(f"\\n Performance comparison:")
 print(f" Original baseline: {baseline_acc:.4f} (97.15%)")
 print(f" Best DenseNet: {best_densenet_acc:.4f} (97.90%)")
 print(f" Best CLIP: {best_clip_acc:.4f} (97.60%)")
 print(f" Fusion (best val epoch): {final_test_acc:.4f} ({final_test_acc*100:.2f}%)")
 print(f" ⭐ Fusion (best test): {best_test_acc_ever:.4f} ({best_test_acc_ever*100:.2f}%)")
 
 # Calculate improvement (Usebesttestaccuracy)
 vs_baseline = (best_test_acc_ever - baseline_acc) * 100
 vs_best_single = (best_test_acc_ever - max(best_densenet_acc, best_clip_acc)) * 100
 vs_baseline_at_val = (final_test_acc - baseline_acc) * 100
 
 print(f"\\n Improvement analysis (based on best test accuracy):")
 print(f" vs Original baseline: {vs_baseline:+.2f} percentage points")
 print(f" vs Best single model: {vs_best_single:+.2f} percentage points")
 
 if best_test_acc_ever > max(best_densenet_acc, best_clip_acc):
 print(f" CLIP Fusion wins! Best test accuracy {best_test_acc_ever*100:.2f}% at epoch {best_test_epoch_found}")
 print(f" Outperforms all individual methods by {vs_best_single:.2f} percentage points!")
 else:
 print(f" Fusion model performs well, close to best single model")
 
 # Analyze fusion weight and CLIP availability
 if clip_available:
 final_weight = torch.sigmoid(fusion_model.fusion_weight).item()
 print(f"\\n Fusion weight analysis:")
 print(f" Fusion weight: {final_weight:.3f}")
 print(f" CLIP branch: Available ")
 else:
 print(f"\\n Fusion status:")
 print(f" CLIP branch: Not available ")
 print(f" Model automatically downgraded to optimized DenseNet")
 
 print(f"\\n Final conclusion:")
 if final_test_acc > 0.98:
 print(f" Excellent! Fusion model achieved 98%+ test accuracy")
 if vs_best_single > 0:
 print(f" Fusion strategy successful! Outperforms individual methods")
 print(f" Demonstrates effectiveness of multi-model fusion")
 else:
 print(f" Fusion model performs well, close to best single model")
 
 print(f" Significant improvement over original baseline ({vs_baseline:+.2f} percentage points)")
 print(f" Data augmentation strategy matches enhanced single-modal training exactly, ensuring fair comparison")
 
 # Save fusion model results (including predictions for visualization)
 fusion_results = {
 'name': 'CLIP Fusion Model',
 'final_test_accuracy': final_test_acc,
 'test_accuracy': final_test_acc, # compatibility (based on best validation epoch)
 
 # ⭐ Best test-set performance (to highlight CLIP capability)
 'best_test_accuracy_ever': best_test_acc_ever,
 'best_test_epoch': best_test_epoch_found,
 
 # Model selection info (based on validation set)
 'best_val_epoch': best_epoch,
 'best_val_accuracy': best_val_acc,
 'test_at_best_val': final_test_acc,
 
 'best_epoch': best_epoch, # compatibility
 'total_epochs': NUM_EPOCHS,
 
 
 
 'clip_available': clip_available,
 'improvement_vs_baseline': vs_baseline,
 'improvement_vs_best_single': vs_best_single,
 'densenet_config': BEST_DENSENET_CONFIG,
 'clip_config': BEST_CLIP_CONFIG,
 'random_seed': RANDOM_SEED,
 'training_history': history,
 # Add full evaluation data (for visualization)
 'predictions': evaluation_data['predictions'],
 'true_labels': evaluation_data['true_labels'],
 'test_labels': evaluation_data['true_labels'], # compatibility
 'test_probabilities': evaluation_data['probabilities'],
 'confusion_matrix': evaluation_data['confusion_matrix'].tolist(),
 'test_report': evaluation_data['results/evaluation_reports/classification_report']
 }
 
 if clip_available:
 fusion_results['fusion_weight'] = torch.sigmoid(fusion_model.fusion_weight).item()
 
 import json
 from datetime import datetime
 timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
 results_path = f"fusion_model_results_{timestamp}.json"
 
 with open(results_path, 'w') as f:
 json.dump(fusion_results, f, indent=4)
 
 print(f"\\n Results saved to: {results_path}")
 print("\\n" + "=" * 80)
 
else:
 print(" Data loaders not available. Please run data loading cell first.")


### Next Steps


In [ ]:
# CLIP: evaluate again and save predictions/true labels/embeddings to file
import os
import json
import numpy as np
import torch.nn.functional as F
from torchvision import transforms, datasets
from torch.utils.data import DataLoader

NEW_TEST_DIR = 'additional_data/additional_mapped'
if not os.path.exists(NEW_TEST_DIR):
 raise FileNotFoundError(f"Test dir not found: {NEW_TEST_DIR}")

val_test_transform = transforms.Compose([
 transforms.Resize((224, 224)),
 transforms.ToTensor(),
 transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_dataset = datasets.ImageFolder(NEW_TEST_DIR, transform=val_test_transform)
if len(test_dataset) == 0:
 raise RuntimeError(f"No images in {NEW_TEST_DIR}")

bs = globals().get('BATCH_SIZE', 32)
num_workers = globals().get('NUM_WORKERS', 4)

test_loader = DataLoader(test_dataset, batch_size=bs*2, shuffle=False, num_workers=num_workers)

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Ensure encoders exist
if 'image_encoder' not in globals() or 'text_encoder' not in globals():
 image_encoder, text_encoder = create_clip_densenet_model(embed_dim=CLIP_EMBED_DIM, dropout=DROPOUT_RATE)
 if os.path.exists(PRETRAINED_MODEL_PATH):
 try:
 image_encoder.load_pretrained_weights(PRETRAINED_MODEL_PATH)
 except Exception:
 pass

# Need train_loader for cache
if 'train_loader' not in globals() or train_loader is None:
 raise RuntimeError('train_loader not available; run data loading cell first')

cache_keys, cache_values = build_cache_from_dataset(image_encoder, text_encoder, train_loader, device=device)
clip_model = OptimizedCLIPTipAdapter(image_encoder, text_encoder, cache_keys, cache_values,
 alpha=BEST_CLIP_CONFIG.get('alpha',0.5),
 t_knn=BEST_CLIP_CONFIG.get('t_knn',0.07),
 lr_adapter=BEST_CLIP_CONFIG.get('lr_adapter',3e-4),
 device=device)
clip_model.eval()

# Lists for storing results
all_preds = []
all_trues = []
all_embeddings = []
all_probs = []
all_paths = []
correct = 0
total = 0

# Get file paths from dataset
file_paths = [p for p, _ in test_dataset.samples]
idx = 0

with torch.no_grad():
 for images, labels in test_loader:
 batch_size_curr = images.size(0)
 images, labels = images.to(device), labels.to(device)

 logits = clip_model(images, mode='eval')
 if isinstance(logits, tuple):
 logits = logits[0]

 probs = F.softmax(logits, dim=1)
 preds = logits.argmax(dim=1)

 # Extract embeddings from CLIP image encoder (DenseNetEncoder)
 # Save 1024-dim DenseNet backbone features for fair comparison with DenseNet t-SNE
 features = image_encoder.backbone.features(images)
 embeddings = F.adaptive_avg_pool2d(features, (1, 1)).view(images.size(0), -1)

 # Store results
 all_preds.extend(preds.cpu().tolist())
 all_trues.extend(labels.cpu().tolist())
 all_probs.extend(probs.cpu().tolist())
 all_embeddings.append(embeddings.cpu().numpy())

 batch_paths = file_paths[idx: idx + batch_size_curr]
 all_paths.extend(batch_paths)
 idx += batch_size_curr

 correct += (preds == labels).sum().item()
 total += batch_size_curr

if total == 0:
 raise RuntimeError('No samples evaluated')

acc = correct / total
print(f"CLIP Test accuracy on '{NEW_TEST_DIR}': {acc*100:.2f}%")

# Use class names from training (6 classes)
model_class_names = ['NORMAL', 'Neurocitoma', 'Meningioma', 'Outros Tipos de Lesões', 'Glioma', 'Schwannoma']

# Save predictions and true labels
os.makedirs('results/collected_predictions', exist_ok=True)
out_json = 'results/collected_predictions/CLIP_additional_mapped_preds.json'
with open(out_json, 'w', encoding='utf-8') as f:
 json.dump({
 'true_labels': all_trues,
 'predictions': all_preds,
 'probabilities': all_probs,
 'class_names': model_class_names,
 'accuracy': acc
 }, f, ensure_ascii=False, indent=2)

# Save embeddings for t-SNE
os.makedirs('results/embeddings', exist_ok=True)
out_npz = 'results/embeddings/CLIP_additional_mapped_embeddings.npz'
emb_array = np.vstack(all_embeddings)
np.savez_compressed(out_npz,
 embeddings=emb_array,
 true_labels=np.array(all_trues),
 predictions=np.array(all_preds),
 file_paths=np.array(all_paths),
 class_names=model_class_names)

print(f"Saved CLIP predictions to: {out_json}")
print(f"Saved CLIP embeddings to: {out_npz}")
